In [ ]:
!python3 -m pip install --user --upgrade pip

In [ ]:
!python3 -m pip install --user --upgrade matplotlib

In [ ]:
!python3 -m pip install --user --upgrade "numpy==1.24.4"

In [ ]:
!pip install ultralytics

In [ ]:
# Lambda Labs Optimized Custom YOLO Hyperparameter Tuning with Genetic Algorithm
import yaml
import os
import sys
import platform
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset
import json
from datetime import datetime
from typing import Dict, Any, Optional, Tuple, List
import random
import pickle
import shutil
import time
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Lambda Labs specific: Set matplotlib backend
plt.switch_backend('Agg')
%matplotlib inline

# Display system info
print(f"🖥️ System Info:")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def configure_ultralytics_datasets_root(root: Path) -> None:
    """Stick Ultralytics datasets_dir to <project_root>."""
    settings.update({"datasets_dir": str(root)})
    print(f"✅  datasets_dir → {settings['datasets_dir']}")

def tidy_dataset_yaml(yaml_path: Path, single_class_name: str | None = None) -> None:
    """Remove 'path:' entry & enforce nc/names consistency."""
    data = yaml.safe_load(yaml_path.read_text())
    if data.pop("path", None) is not None:
        print("🔁  Removed stale 'path:' key from dataset YAML")
    if single_class_name:
        data["nc"] = 1
        data["names"] = [single_class_name]
    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
    print(f"✅  Dataset YAML saved: {yaml_path.relative_to(Path().resolve())}")
    
def load_config(cfg_path: str) -> dict:
    cfg = yaml.safe_load(Path(cfg_path).read_text())
    if not cfg:
        raise RuntimeError("Empty configuration file")
    return cfg

In [ ]:
project_root = Path().resolve()
configure_ultralytics_datasets_root(project_root)

In [ ]:
CONFIG_PATH = project_root / 'config.yaml'
print(os.path.exists(CONFIG_PATH))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.  Load training configuration
# ──────────────────────────────────────────────────────────────────────────────
cfg = load_config(CONFIG_PATH)

configs = cfg.get("powerline_inference",{})

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 5.  Build inference arguments
# ──────────────────────────────────────────────────────────────────────────────
args = {
    "source"   : configs.get("source",),
    "project": configs.get("project",),  # Base directory for all experiments
    "name": configs.get("name",),
    "save": configs.get("save",),
    "save_conf": configs.get("save_conf",),
    "conf": configs.get("conf",),
    "half": configs.get("half",),
    "device": cfg.get("device",),
    "imgsz": configs.get("imgsz",)
    
}
print("📝  Inference arguments:")
print(json.dumps(args, indent=4), "\n")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 6.  Initialise model
# ──────────────────────────────────────────────────────────────────────────────
print("🚀  Initialising model:",configs.get("model",))
model = YOLO(configs.get("model",))
print("✅  Model ready\n")

In [ ]:
results = model.predict(**args)